# 03: Model Selection and VRAM Budgeting

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/finetuning/03_model_selection_and_vram_budgeting.ipynb)

Good finetuning starts with hardware realism. This notebook shows how to choose open models for T4, L4, and A100 class runtimes, why QLoRA is the default on Colab Pro, and how to build a rough budgeting helper before you burn time on failed runs.

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## Learning goals

- compare T4, L4, and A100 constraints from a finetuning perspective
- distinguish QLoRA, LoRA, and dense tuning in practical memory terms
- estimate rough VRAM needs for 3B, 4B, and 8B class models
- reason about sequence length, batch size, and gradient accumulation together
- validate a budget estimate against a real model load

In [ ]:
import math
import statistics
from pathlib import Path

ARTIFACT_DIR = Path("artifacts") / "notebook_03_model_selection_vram"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def print_table(rows, headers=None):
    rows = list(rows)
    if not rows:
        print("No rows")
        return

    if headers is None:
        headers = list(rows[0].keys())

    widths = []
    for header in headers:
        longest = max(len(str(row.get(header, ""))) for row in rows)
        widths.append(max(len(str(header)), longest))

    header_line = " | ".join(str(header).ljust(widths[index]) for index, header in enumerate(headers))
    divider = "-+-".join("-" * width for width in widths)
    print(header_line)
    print(divider)
    for row in rows:
        print(" | ".join(str(row.get(header, "")).ljust(widths[index]) for index, header in enumerate(headers)))

def clipped(value, low=0.0, high=1.0):
    return max(low, min(high, value))

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Why hardware-aware choice matters

Model choice is never only about benchmark scores. It is about whether the full training recipe fits:

- base weights
- adapter parameters
- optimizer state
- gradients
- activations
- safety margin for fragmentation and notebook overhead

Ignoring those pieces is how otherwise reasonable experiments turn into repeated out-of-memory failures.

In [ ]:
hardware_profiles = [
    {
        "gpu": "T4",
        "vram_gb": 16,
        "comfort_zone": "3B to 4B QLoRA",
        "risky_zone": "8B with longer contexts or aggressive batches",
    },
    {
        "gpu": "L4",
        "vram_gb": 24,
        "comfort_zone": "7B to 8B QLoRA",
        "risky_zone": "dense tuning or very long contexts",
    },
    {
        "gpu": "A100",
        "vram_gb": 40,
        "comfort_zone": "larger ablations and more context headroom",
        "risky_zone": "wasting budget because the recipe was never scoped",
    },
]

print_table(hardware_profiles, headers=["gpu", "vram_gb", "comfort_zone", "risky_zone"])

## Open-source model classes for this course

The course stays conservative and open-source. The point is not to chase the biggest available checkpoint. The point is to pick a model family that fits iteration speed, eval discipline, and your actual GPU.

In [ ]:
model_catalog = [
    {
        "class": "3B",
        "example_family": "Qwen or Gemma 3B style instruct model",
        "sweet_spot_gpu": "T4",
        "best_use": "first runs, shorter contexts, cheap ablations",
    },
    {
        "class": "4B",
        "example_family": "Qwen 4B style instruct model",
        "sweet_spot_gpu": "T4 or L4",
        "best_use": "balanced quality vs memory",
    },
    {
        "class": "8B",
        "example_family": "Qwen or Llama style 7B to 8B instruct model",
        "sweet_spot_gpu": "L4 or A100",
        "best_use": "stronger quality when QLoRA still fits",
    },
]

print_table(model_catalog, headers=["class", "example_family", "sweet_spot_gpu", "best_use"])

## QLoRA vs LoRA vs dense tuning

These methods differ in more than memory:

- QLoRA keeps frozen base weights in 4-bit form and trains adapters
- LoRA keeps frozen base weights at higher precision and trains adapters
- dense tuning updates the full model

The memory difference is exactly why QLoRA became the default Colab recipe.

In [ ]:
tuning_methods = [
    {
        "method": "QLoRA",
        "base_weights_precision": "4-bit",
        "trainable_fraction": "tiny adapter subset",
        "memory_pressure": "lowest practical default",
        "when_to_use": "default on Colab Pro",
    },
    {
        "method": "LoRA",
        "base_weights_precision": "16-bit or similar",
        "trainable_fraction": "tiny adapter subset",
        "memory_pressure": "higher than QLoRA",
        "when_to_use": "when quantized loading is not desirable",
    },
    {
        "method": "Dense tuning",
        "base_weights_precision": "full training precision",
        "trainable_fraction": "all parameters",
        "memory_pressure": "very high",
        "when_to_use": "rare, carefully justified cases",
    },
]

print_table(tuning_methods, headers=["method", "base_weights_precision", "trainable_fraction", "memory_pressure", "when_to_use"])

## Where VRAM goes during training

A rough training memory budget includes:

1. base model weights
2. adapter weights
3. optimizer state for trainable parameters
4. gradients
5. activations, which grow with sequence length and effective batch

That last item is why runs that load successfully can still fail during backprop.

In [ ]:
MODEL_CONFIGS = {
    "3B": {"params_b": 3.0, "layers": 28, "hidden_size": 3072},
    "4B": {"params_b": 4.0, "layers": 32, "hidden_size": 3584},
    "8B": {"params_b": 8.0, "layers": 36, "hidden_size": 4096},
}

METHOD_CONFIGS = {
    "qlora": {"weight_bytes": 0.60, "trainable_fraction": 0.0020, "optimizer_bytes": 8.0, "gradient_bytes": 2.0},
    "lora": {"weight_bytes": 2.00, "trainable_fraction": 0.0020, "optimizer_bytes": 8.0, "gradient_bytes": 2.0},
    "dense": {"weight_bytes": 2.00, "trainable_fraction": 1.0000, "optimizer_bytes": 8.0, "gradient_bytes": 2.0},
}

def estimate_activations_gb(layers, hidden_size, seq_length, micro_batch_size):
    bytes_total = layers * hidden_size * seq_length * micro_batch_size * 10.0
    return bytes_total / (1024 ** 3)

def estimate_budget(model_key, method_key, seq_length=2048, micro_batch_size=1, gradient_accumulation=8):
    model_cfg = MODEL_CONFIGS[model_key]
    method_cfg = METHOD_CONFIGS[method_key]
    params = model_cfg["params_b"] * 1_000_000_000

    base_weights_gb = params * method_cfg["weight_bytes"] / (1024 ** 3)
    trainable_params = params * method_cfg["trainable_fraction"]
    adapter_or_trainable_gb = trainable_params * 2.0 / (1024 ** 3)
    optimizer_gb = trainable_params * method_cfg["optimizer_bytes"] / (1024 ** 3)
    gradients_gb = trainable_params * method_cfg["gradient_bytes"] / (1024 ** 3)
    activations_gb = estimate_activations_gb(
        model_cfg["layers"],
        model_cfg["hidden_size"],
        seq_length,
        micro_batch_size,
    )
    fragmentation_gb = 1.2
    total_gb = base_weights_gb + adapter_or_trainable_gb + optimizer_gb + gradients_gb + activations_gb + fragmentation_gb

    return {
        "base_weights_gb": round(base_weights_gb, 2),
        "trainable_weights_gb": round(adapter_or_trainable_gb, 2),
        "optimizer_gb": round(optimizer_gb, 2),
        "gradients_gb": round(gradients_gb, 2),
        "activations_gb": round(activations_gb, 2),
        "fragmentation_gb": fragmentation_gb,
        "total_gb": round(total_gb, 2),
        "effective_batch": micro_batch_size * gradient_accumulation,
    }

example_budget = estimate_budget("4B", "qlora", seq_length=2048, micro_batch_size=1, gradient_accumulation=8)
print(example_budget)

## Budgeting heuristics are meant to guide decisions, not predict exact bytes

Framework overhead, attention kernels, tokenizer behavior, and implementation details all shift real usage. But a rough budget is still valuable because it tells you:

- what obviously will not fit
- which settings are near the edge
- where your main memory pressure is coming from

In [ ]:
training_profiles = [
    {"label": "safe_short_context", "seq_length": 1024, "micro_batch_size": 1, "gradient_accumulation": 8},
    {"label": "balanced_default", "seq_length": 2048, "micro_batch_size": 1, "gradient_accumulation": 8},
    {"label": "longer_context_push", "seq_length": 3072, "micro_batch_size": 1, "gradient_accumulation": 8},
]

rows = []
for profile in training_profiles:
    budget = estimate_budget(
        "4B",
        "qlora",
        seq_length=profile["seq_length"],
        micro_batch_size=profile["micro_batch_size"],
        gradient_accumulation=profile["gradient_accumulation"],
    )
    rows.append({
        "label": profile["label"],
        "seq_length": profile["seq_length"],
        "micro_batch_size": profile["micro_batch_size"],
        "effective_batch": budget["effective_batch"],
        "total_gb_estimate": budget["total_gb"],
    })

print_table(rows, headers=["label", "seq_length", "micro_batch_size", "effective_batch", "total_gb_estimate"])

## Compare realistic setups across hardware

Now we can ask a useful question: which model and method combinations are likely to fit on common Colab GPUs with a modest safety margin?

In [ ]:
hardware_budget = {"T4": 16, "L4": 24, "A100": 40}

rows = []
for model_key in ["3B", "4B", "8B"]:
    for method_key in ["qlora", "lora", "dense"]:
        budget = estimate_budget(model_key, method_key, seq_length=2048, micro_batch_size=1, gradient_accumulation=8)
        for gpu_name, vram_gb in hardware_budget.items():
            fits = budget["total_gb"] <= (vram_gb - 1.5)
            rows.append({
                "model": model_key,
                "method": method_key,
                "gpu": gpu_name,
                "estimate_gb": budget["total_gb"],
                "fits_with_margin": fits,
            })

print_table(rows, headers=["model", "method", "gpu", "estimate_gb", "fits_with_margin"])

## What this table is telling us

The pattern should be clear:

- 3B and 4B QLoRA are the comfortable T4 choices
- 8B QLoRA is usually an L4 move, not a casual T4 default
- full dense tuning escalates memory so sharply that Colab should not be your first assumption

In other words, bigger is not better if it kills iteration speed.

In [ ]:
def recommend_configuration(gpu_name, priority):
    if gpu_name == "T4":
        if priority == "safest":
            return {"model": "3B", "method": "qlora", "seq_length": 1024}
        if priority == "balanced":
            return {"model": "4B", "method": "qlora", "seq_length": 2048}
        return {"model": "4B", "method": "qlora", "seq_length": 1536}
    if gpu_name == "L4":
        if priority == "safest":
            return {"model": "4B", "method": "qlora", "seq_length": 2048}
        if priority == "balanced":
            return {"model": "8B", "method": "qlora", "seq_length": 2048}
        return {"model": "8B", "method": "qlora", "seq_length": 3072}
    if priority == "safest":
        return {"model": "4B", "method": "qlora", "seq_length": 2048}
    if priority == "balanced":
        return {"model": "8B", "method": "qlora", "seq_length": 3072}
    return {"model": "8B", "method": "lora", "seq_length": 4096}

rows = []
for gpu_name in ["T4", "L4", "A100"]:
    for priority in ["safest", "balanced", "stretch"]:
        recommendation = recommend_configuration(gpu_name, priority)
        budget = estimate_budget(
            recommendation["model"],
            recommendation["method"],
            seq_length=recommendation["seq_length"],
            micro_batch_size=1,
            gradient_accumulation=8,
        )
        rows.append({
            "gpu": gpu_name,
            "priority": priority,
            "model": recommendation["model"],
            "method": recommendation["method"],
            "seq_length": recommendation["seq_length"],
            "estimate_gb": budget["total_gb"],
        })

print_table(rows, headers=["gpu", "priority", "model", "method", "seq_length", "estimate_gb"])

## Context length is a budget decision

Longer context is useful, but it is not free. On limited VRAM, sequence length often becomes the main source of activation growth.

In [ ]:
rows = []
for seq_length in [1024, 1536, 2048, 3072, 4096]:
    budget = estimate_budget("8B", "qlora", seq_length=seq_length, micro_batch_size=1, gradient_accumulation=8)
    rows.append({
        "model": "8B",
        "method": "qlora",
        "seq_length": seq_length,
        "activation_gb": budget["activations_gb"],
        "total_gb": budget["total_gb"],
    })

print_table(rows, headers=["model", "method", "seq_length", "activation_gb", "total_gb"])

## Batch size is a budget decision too

When memory is tight, gradient accumulation is often the safer knob to turn. It increases effective batch without demanding that everything fit at once.

In [ ]:
rows = []
for micro_batch_size in [1, 2, 4]:
    budget = estimate_budget("4B", "qlora", seq_length=2048, micro_batch_size=micro_batch_size, gradient_accumulation=8)
    rows.append({
        "micro_batch_size": micro_batch_size,
        "effective_batch": budget["effective_batch"],
        "activation_gb": budget["activations_gb"],
        "total_gb": budget["total_gb"],
    })

print_table(rows, headers=["micro_batch_size", "effective_batch", "activation_gb", "total_gb"])

## Dense tuning is the exception, not the doctrine

Dense tuning can be appropriate when you have unusual hardware, unusual requirements, or a very specific research reason. But on Colab Pro, it should start as a justification problem, not a default assumption.

In [ ]:
checkpoint_rows = []
for model_key, cfg in MODEL_CONFIGS.items():
    dense_checkpoint_gb = round(cfg["params_b"] * 1_000_000_000 * 2.0 / (1024 ** 3), 2)
    adapter_checkpoint_mb = round(cfg["params_b"] * 1_000_000_000 * 0.002 * 2.0 / (1024 ** 2), 1)
    checkpoint_rows.append({
        "model": model_key,
        "dense_checkpoint_gb": dense_checkpoint_gb,
        "adapter_checkpoint_mb": adapter_checkpoint_mb,
    })

print_table(checkpoint_rows, headers=["model", "dense_checkpoint_gb", "adapter_checkpoint_mb"])

## Real load verification with the shared setup

Budgeting gets more valuable when you compare it with a real runtime. The next cell loads the default course model and adapter recipe using the shared setup.

In [ ]:
# --- Google Colab Pro Fine-Tuning Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## Compare estimated and observed memory

The exact number will depend on runtime state, allocator behavior, and cached kernels. Still, comparing estimate and observation helps calibrate our mental model.

In [ ]:
observed_allocated_gb = round(torch.cuda.memory_allocated() / (1024 ** 3), 2) if torch.cuda.is_available() else 0.0
observed_reserved_gb = round(torch.cuda.memory_reserved() / (1024 ** 3), 2) if torch.cuda.is_available() else 0.0

estimated_default = estimate_budget("8B", "qlora", seq_length=MAX_SEQ_LENGTH, micro_batch_size=1, gradient_accumulation=8)

comparison_rows = [
    {"metric": "estimated_total_gb", "value": estimated_default["total_gb"]},
    {"metric": "observed_allocated_gb", "value": observed_allocated_gb},
    {"metric": "observed_reserved_gb", "value": observed_reserved_gb},
]

print_table(comparison_rows, headers=["metric", "value"])
print()
print("Loaded model:", MODEL_NAME)
print("Reminder: observed memory after loading is lower than full training peak because activations grow during forward and backward passes.")

## Takeaways

- model selection should begin with available VRAM, not with wishful thinking
- QLoRA is the default because it keeps the base model cheap enough to fit
- 3B and 4B are excellent T4 starting points; 8B is usually an L4 move
- sequence length and micro-batch size can matter as much as checkpoint size
- rough budgeting is not exact, but it prevents many bad experiments

In the next notebook, we move from hardware to data formatting, chat templates, and loss masking.